In [92]:
import matplotlib.pyplot as plt
import numpy.typing as npt
from pathlib import Path
from typing import Tuple
import pandas as pd
import numpy as np
import re
import matplotlib.ticker as ticker

# Ahora importa el módulo
from RRAM.Representate import config_ax, setup_paper_plt

setup_paper_plt(plt, latex=True, scaling=2)

In [93]:
# Rutas de los archivos
ruta_datos = (Path.cwd() / "Datos_Experimentales" / "Medidas_Experimentales_RRAM")
ruta_figuras = Path.cwd() / "Datos_Experimentales" / "V_Set"

ruta_figuras.mkdir(exist_ok=True)

# Listar todos los archivos
archivos = []
for archivo in ruta_datos.glob("Cycle_p_*.txt"):
    # Buscar el número dentro del nombre (después de "Cycle_p_")
    match = re.fullmatch(r"Cycle_p_(\d{3,4})\.txt", archivo.name)
    # match = re.fullmatch(r"Cycle_p_(1000|[1-9][0-9]{0,2})\.txt", archivo.name)
    if match:
        numero = int(match.group(1))  # convertir a entero
        if 1 <= numero <= 1301:  # aplicar filtro de rango
            archivos.append(archivo)

print(f"Archivos encontrados: {len(archivos)}")
print(archivos)

Archivos encontrados: 1202
[WindowsPath('c:/Users/Usuario/Documents/GitHub/RRAM_Simulation/Datos_Experimentales/Medidas_Experimentales_RRAM/Cycle_p_100.txt'), WindowsPath('c:/Users/Usuario/Documents/GitHub/RRAM_Simulation/Datos_Experimentales/Medidas_Experimentales_RRAM/Cycle_p_1000.txt'), WindowsPath('c:/Users/Usuario/Documents/GitHub/RRAM_Simulation/Datos_Experimentales/Medidas_Experimentales_RRAM/Cycle_p_1001.txt'), WindowsPath('c:/Users/Usuario/Documents/GitHub/RRAM_Simulation/Datos_Experimentales/Medidas_Experimentales_RRAM/Cycle_p_1002.txt'), WindowsPath('c:/Users/Usuario/Documents/GitHub/RRAM_Simulation/Datos_Experimentales/Medidas_Experimentales_RRAM/Cycle_p_1003.txt'), WindowsPath('c:/Users/Usuario/Documents/GitHub/RRAM_Simulation/Datos_Experimentales/Medidas_Experimentales_RRAM/Cycle_p_1004.txt'), WindowsPath('c:/Users/Usuario/Documents/GitHub/RRAM_Simulation/Datos_Experimentales/Medidas_Experimentales_RRAM/Cycle_p_1005.txt'), WindowsPath('c:/Users/Usuario/Documents/GitHub/RR

In [94]:
def plot_iv_and_derivative(voltaje, intensidad, dI_dV, top_indices, ruta_figuras):
    """
    Plots the I-V curve and its derivative, highlighting specific points with the highest derivative values.
    Parameters:
    -----------
    V_set : array-like
        Array of voltage values (x-axis data).
    I_set : array-like
        Array of current values corresponding to the voltage values (y-axis data for the I-V curve).
    dI_dV : array-like
        Array of derivative values (y-axis data for the derivative plot).
    top_indices : list of int
        Indices of the points with the highest derivative values to be highlighted on the plots.
    ruta_figuras : str
        File path where the generated figure will be saved.
    Returns:
    --------
    None
        The function saves the generated plot as an image file at the specified location.
    Notes:
    ------
    - The function creates a figure with two subplots:
        1. The I-V curve with logarithmic scaling on the y-axis.
        2. The derivative of the I-V curve.
    - Points with the highest derivative values are highlighted in red on both subplots.
    - The figure is saved as an image file in the specified path with a resolution of 300 dpi.
    """
    
    # Crear una figura con dos subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # === Configuración de estilo LaTeX y plot ===
    setup_paper_plt(plt, latex=True, scaling=2)

    config_ax(ax1)
    config_ax(ax2)

    # En el primer subplot, graficar la curva I-V
    ax1.plot(voltaje, intensidad, "b-", linewidth=2)
    ax1.set_xlabel(r"Voltaje (\si{\volt})")
    ax1.set_ylabel(r"Corriente (\si{\ampere})")
    ax1.set_title("Curva I-V")
    ax1.set_yscale("log")
    ax1.grid(True)

    # Solo marcar los puntos con mayor derivada en el primer subplot
    for idx in top_indices:
        ax1.plot(voltaje[idx], intensidad[idx], "o", markersize=6, color="red")
        ax1.annotate(
            f"{voltaje[idx]:.2f} V",
            (voltaje[idx], intensidad[idx]),
            textcoords="offset points",
            xytext=(-35, -5),
            ha="center",
            fontsize=16,
            fontweight="bold",
        )

    # En el segundo subplot, graficar la derivada de la corriente respecto al voltaje
    ax2.plot(voltaje, dI_dV, "-", linewidth=2, color = "blue")
    ax2.set_xlabel(r"Voltaje (\si{\volt})")
    ax2.set_ylabel(r"$\dv{I}{V}$ (\si{\ampere/\volt})")
    ax2.set_title("Derivada de la curva I-V")
    ax2.grid(True)

    # Solo marcar los puntos con mayor derivada en el segundo subplot
    for idx in top_indices:
        ax2.plot(voltaje[idx], dI_dV[idx], "o", markersize=6, color="red")
        ax2.annotate(
            f"{voltaje[idx]:.2f} V",
            (voltaje[idx], dI_dV[idx]),
            textcoords="offset points",
            xytext=(-25, -5),
            ha="center",
            fontsize=16,
            fontweight="bold",
        )
        
    fig.savefig(str(ruta_figuras) + ".pdf", dpi=300, bbox_inches="tight")  # type: ignore
    plt.close(fig)
    
    # Crear figura combinada con eje secundario
    fig_combinada, ax_comb_1 = plt.subplots(figsize=(12, 9))
    
    setup_paper_plt(plt, latex=True, scaling=2)
    config_ax(ax_comb_1)
    
    # ---- Eje izquierdo (Corriente) ----
    ax_comb_1.semilogy(voltaje, intensidad, "o-", color="blue", label=r"Corriente (\si{\ampere})")
    ax_comb_1.set_xlabel(r"Voltaje (\si{\volt})")
    ax_comb_1.set_ylabel(r"Corriente (\si{\ampere})", color="blue")
    ax_comb_1.tick_params(axis="y", labelcolor="blue")

    # Solo marcar los puntos con mayor derivada en el primer subplot
    for idx in top_indices:
        ax_comb_1.plot(voltaje[idx], intensidad[idx], "o", markersize=6, color="red")
        ax_comb_1.annotate(
            f"{voltaje[idx]:.2f} V",
            (voltaje[idx], intensidad[idx]),
            textcoords="offset points",
            xytext=(-35, -5),
            ha="center",
            fontsize=16,
            fontweight="bold",
        )
    
    # ---- Eje derecho (Derivada) ----
    ax_comb_2 = ax_comb_1.twinx()  # Crear un eje Y secundario
    config_ax(ax2)
    ax_comb_2.plot(
        voltaje, dI_dV, "o-", color="orange", label=r"$\dv{I}{V}$ (\si{\ampere/\volt})"
    )
    ax_comb_2.set_ylabel(r"$\dv{I}{V}$ (\si{\ampere/\volt})", color="orange")
    ax_comb_2.tick_params(axis="y", labelcolor="orange")

    # Solo marcar los puntos con mayor derivada en el segundo subplot
    for idx in top_indices:
        ax_comb_2.plot(voltaje[idx], dI_dV[idx], "o", markersize=6, color="green")
        ax_comb_2.annotate(
            f"{voltaje[idx]:.2f} V",
            (voltaje[idx], dI_dV[idx]),
            textcoords="offset points",
            xytext=(-25, -5),
            ha="center",
            fontsize=16,
            fontweight="bold",
        )
    
    # Leyendas combinadas
    lines1, labels1 = ax_comb_1.get_legend_handles_labels()
    lines2, labels2 = ax_comb_2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="best")
    plt.title("I-V y su Derivada")
    
    fig_combinada.savefig(
        str(ruta_figuras) + "_combinado.pdf", dpi=300, bbox_inches="tight"
    )  # type: ignore
    plt.close(fig_combinada)

In [95]:
def obtener_V_set_Derivada(ruta_datos, num_valores_maximos=3):
    """
    Analiza un archivo de datos experimentales para calcular la derivada de la corriente 
    con respecto al voltaje, identifica los valores máximos de la derivada y genera una 
    gráfica de los datos.
    Args:
        ruta_datos (Path): Ruta al archivo que contiene los datos experimentales. 
            El archivo debe existir y contener tres columnas: "Voltaje", "Corriente" y "tiempo".
        num_valores_maximos (int, opcional): Número de valores máximos de la derivada a extraer. 
            Por defecto es 5.
    Returns:
        dict: Un diccionario con las siguientes claves:
            - "top_derivadas" (np.ndarray): Los valores máximos de la derivada numérica.
            - "top_voltajes" (np.ndarray): Los voltajes correspondientes a los valores máximos 
              de la derivada.
    Raises:
        FileNotFoundError: Si el archivo especificado en `ruta_datos` no existe.
    Notas:
        - La función filtra los datos para considerar solo las primeras 111 filas del archivo.
        - Genera una gráfica que muestra la curva IV y la derivada, y la guarda en una 
          subcarpeta "Datos_Experimentales/V_Set" con un nombre basado en el archivo de entrada.
    """
    # Verificar que el archivo existe
    if not ruta_datos.exists():
        raise FileNotFoundError(
            f"El archivo {ruta_datos} no existe. Verificar la ruta."
        )

    # Cargar datos experimentales
    data_set = np.loadtxt(ruta_datos)

    # Crear el DataFrame
    df_set = pd.DataFrame(data_set, columns=["Voltaje", "Corriente", "tiempo"])
    df_set = df_set.iloc[0:110]  # Seleccionar solo las primeras 111 filas
    
    # Combinación de ambos filtros
    # Primero, resetea los índices después del filtrado
    df_set_filtrado = df_set.loc[(df_set['Voltaje'] >= 0.3) & (df_set['Voltaje'] <= 0.8)].reset_index(drop=True)

    # Calcular la derivada numérica de la corriente con respecto al voltaje
    dI_dV = np.gradient(np.log10(df_set_filtrado["Corriente"]), df_set_filtrado['Voltaje'])

    # Extraer los índices de los valores máximos
    indices_maximos = np.argsort(dI_dV)[-num_valores_maximos:]

    # Extraer los valores máximos y sus voltajes correspondientes
    top_derivadas = dI_dV[indices_maximos]
    top_voltages = df_set_filtrado["Voltaje"].values[indices_maximos]
    
    # V_set = df_set_filtrado["Voltaje"]
    # I_set = df_set_filtrado["Corriente"]
    
    ruta_figuras = (
        Path.cwd()
        / "Datos_Experimentales"
        / "V_Set"
        / f"{ruta_datos.stem}"
    )
    
    # plot_iv_and_derivative(V_set, I_set, dI_dV, indices_maximos, ruta_figuras)
    
    # Retornar resultados en un diccionario
    return {
        "top_derivadas": top_derivadas,
        "top_voltajes": top_voltages,
    }

In [96]:
def plot_elbow_method(
    ruta_datos: Path,
    voltage: npt.NDArray,
    current: npt.NDArray,
    elbow_voltage: float,
    elbow_current: float,
) -> None:
    """
    Representa gráficamente el método del "punto codo" en una curva Voltaje-Corriente.

    Args:
        voltage (npt.NDArray): Array de voltajes.
        current (npt.NDArray): Array de corrientes.

    Returns:
        None: Muestra la gráfica.
    """

    ruta_figuras = (
        Path.cwd()
        / "Datos_Experimentales"
        / "V_Set"
        / f"{ruta_datos.stem}"
    )
    
    # Determinar el rango de la primera mitad
    max_c_idx = np.where(current == current.max())[0][0]
    v_first_half = voltage[: max_c_idx + 1]
    i_first_half = current[: max_c_idx + 1]

    # Puntos inicial y final de la primera mitad
    start_point = np.array([v_first_half[0], i_first_half[0]])
    end_point = np.array([v_first_half[-1], i_first_half[-1]])

    # --- Gráfica ---
    # Crear una figura con dos subplots
    fig, ax1 = plt.subplots(figsize=(12, 9))
    
    setup_paper_plt(plt, latex=True, scaling=2)
    
    # === Configuración de estilo LaTeX y plot ===
    config_ax(ax1)


    # Curva completa
    ax1.plot(voltage, current, label="Curva V-I", color="blue")

    # Recta de referencia
    ax1.plot(
        [start_point[0], end_point[0]],
        [start_point[1], end_point[1]],
        "--",
        color="orange",
        label="Línea de referencia",
    )

    # Punto codo
    ax1.scatter(
        elbow_voltage,
        elbow_current,
        color="red",
        s=100,
        zorder=5,
        label="Punto Codo (Elbow)",
    )
    
    # Mejorar visualización
    ax1.set_xlabel(r"Voltaje (\si{\volt})")
    ax1.set_ylabel(r"Corriente (\si{\ampere})")
    ax1.set_title("Método del Punto Codo en Curva I-V")
    ax1.legend()

    # fig.savefig(str(ruta_figuras) + "_elbow.pdf", dpi=300, bbox_inches="tight")
    plt.close(fig)


In [97]:
def obtener_V_set_elbow(ruta_datos, **kwargs) -> Tuple[float, float, int]:
    """
    Calculate the elbow point in a voltage-current curve.
    This function identifies the elbow point in a voltage-current curve by 
    finding the point with the maximum perpendicular distance from the line 
    connecting the start and end points of the first half of the curve.
    Args:
        ruta_datos (Path): Path to the file containing experimental data. The file 
            should contain three columns: "Voltaje", "Corriente", and "tiempo".
        **kwargs: Additional keyword arguments (not used in the current implementation).
    Returns:
        Tuple[float, float, int]: A tuple containing:
            - elbow_voltage (float): The voltage value at the elbow point.
            - elbow_current (float): The current value at the elbow point.
            - max_dist_idx (int): The index of the elbow point in the data.
    Raises:
        FileNotFoundError: If the file specified by `ruta_datos` does not exist.
    Notes:
        - The function assumes that the input file exists and is formatted correctly.
        - Only the first 110 rows of the data are considered for analysis.
        - The function plots the elbow point on the voltage-current curve using 
          the `plot_elbow_method` function.
    """
    # Verificar que el archivo existe
    if not ruta_datos.exists():
        raise FileNotFoundError(
            f"El archivo {ruta_datos} no existe. Verificar la ruta."
        )

    # Cargar datos experimentales
    data_set = np.loadtxt(ruta_datos)

    # Crear el DataFrame
    df_set = pd.DataFrame(data_set, columns=["Voltaje", "Corriente", "tiempo"])
    df_set = df_set.iloc[0:110]  # Seleccionar solo las primeras 111 filas

    voltage = df_set["Voltaje"]
    current = df_set["Corriente"]

    # Verificar si hay datos
    if len(voltage) == 0 or len(current) == 0:
        raise ValueError("Los arrays de voltaje o corriente están vacíos")

    max_c = current.max()
    max_c_idx = np.where(current == max_c)[0][0]

    # Verificar si se puede dividir el array
    if max_c_idx == 0:
        raise ValueError("No hay suficientes datos para dividir el array")
    
    # print(f"Max current: {max_c} at index {max_c_idx}")

    # Determinar el rango de la primera mitad
    v_first_half = voltage[: max_c_idx + 1]
    i_first_half = current[: max_c_idx + 1]

    # Verificar que los arrays resultantes no estén vacíos
    if len(v_first_half) == 0 or len(i_first_half) == 0:
        raise ValueError("Los arrays resultantes están vacíos")

    start_point = np.array([v_first_half.iloc[0], i_first_half.iloc[0]])
    end_point = np.array([v_first_half.iloc[-1], i_first_half.iloc[-1]])

    distances = np.abs(
        (end_point[0] - start_point[0]) * (start_point[1] - i_first_half)
        - (start_point[0] - v_first_half) * (end_point[1] - start_point[1])
    ) / np.linalg.norm(end_point - start_point)

    max_dist_idx = distances.argmax()
    
    elbow_voltage = float(v_first_half.iloc[max_dist_idx])
    elbow_current = float(i_first_half.iloc[max_dist_idx])
    max_dist_idx = int(max_dist_idx)
    
    # plot_elbow_method(ruta_datos, voltage.to_numpy(), current.to_numpy(), elbow_voltage, elbow_current)

    return elbow_voltage, elbow_current, max_dist_idx


In [98]:
# Método de la derivada
for datos in archivos:   
    resultados = obtener_V_set_Derivada(datos, num_valores_maximos=3)
    elbow_voltage, elbow_current, max_dist_idx = obtener_V_set_elbow(datos)
    
    # Encuentra el índice del voltaje más bajo
    idx_min_voltaje = np.argmin(resultados["top_voltajes"])
    voltaje_min = resultados["top_voltajes"][idx_min_voltaje]
    derivada_min = resultados["top_derivadas"][idx_min_voltaje]

    # Guardo los resultados en un único archivo de texto
    with open(ruta_figuras / "V_set_experimental.txt", "a") as f:
        f.write(f"{datos.stem}\t{voltaje_min:.4f}\t{elbow_voltage:.4f}\n")

In [99]:
results_path = ruta_figuras / "V_set_experimental.txt"

resultados_txt = np.genfromtxt(
    results_path,
    dtype=[
        ("Archivo", "U50"),  # String Unicode de máx 50 caracteres
        ("V_set_derivada", "f8"),  # Float de 64 bits
        ("V_set_elbow", "f8"),
    ],
    encoding="utf-8",
    names=["Archivo", "V_set_derivada_V", "V_set_elbow_V"],
)

# Crear un DataFrame para una mejor visualización
df_resultados = pd.DataFrame(resultados_txt)

df_resultados["Numero"] = (
    df_resultados["Archivo"].str.extract(r"Cycle_p_(\d+)", expand=False).astype(int)
)

In [100]:
# Obtener V_set de los archivos de simulación
ruta_archivos_simulacion = Path.cwd() / "logs"

# Listar todos los archivos
resultados = []

for archivo in ruta_archivos_simulacion.glob("log_simulacion_*.log"):
    with open(archivo, "r", encoding="utf-8") as f:
        contenido = f.read()

    # Buscar el número después de "voltaje:"
    pattern = r"El filamento\s+(\d)\s+se ha creado en el voltaje\s+(-?\d*\.?\d+(?:[eE][-+]?\d+)?)"
    matches = re.findall(pattern, contenido, re.IGNORECASE)
    
    if match:
        voltajes = [float(v) for _, v in matches]
        voltajes_str = [f"{v:.4f}" for v in voltajes]

        linea = [archivo.name] + voltajes_str
        if len(voltajes) == 2:
            with open(ruta_figuras / "V_set_simulacion_2_CF.txt", "a") as f:
                f.write("\t".join(linea) + "\n")
        else:
            print(f"El archivo {archivo.name} no tiene 2 filamentos creados, tiene {len(voltajes)}")

results_path = ruta_figuras / "V_set_simulacion_2_CF.txt"

resultados_txt = np.genfromtxt(
    results_path,
    dtype=[
        ("Archivo", "U50"),  # String Unicode de máx 50 caracteres
        ("V_creacion_1", "f8"),  # Float de 64 bits
        ("V_creacion_2", "f8"),  # Float de 64 bits
        # ("V_creacion_3", "f8"),  # Float de 64 bits
        # ("V_creacion_4", "f8"),  # Float de 64 bits
    ],
    encoding="utf-8",
    names=[
        "Archivo",
        "V_creacion_1",
        "V_creacion_2",
        # "V_creacion_3",
        # "V_creacion_4",
    ],  # NO se refiere a que se ha creado el filamento 1 2 3 o 4, sino al orden en que aparecen en el log
)

# Crear un DataFrame para una mejor visualización
df_resultados_sim = pd.DataFrame(resultados_txt)

df_resultados_sim["Numero"] = (
    df_resultados_sim["Archivo"]
    .str.extract(r"log_simulacion_(\d+)", expand=False)
    .astype(int)
)

El archivo log_simulacion_44.log no tiene 2 filamentos creados, tiene 1
El archivo log_simulacion_88.log no tiene 2 filamentos creados, tiene 1
El archivo log_simulacion_9.log no tiene 2 filamentos creados, tiene 1
El archivo log_simulacion_98.log no tiene 2 filamentos creados, tiene 1


In [101]:
plt.rc("xtick", labelsize=36)
plt.rc("ytick", labelsize=36)

fig, ax = plt.subplots(figsize=(12, 9))

# TODO: tengo q revisar cómo se extrae el número de curva de cada archivo
# === Configuración de estilo LaTeX y plot ===
setup_paper_plt(plt, latex=True, scaling=3)
config_ax(ax)

# ---------- EJE X ----------
# Marcas exactas: -1.5 -1.0 −0.5 0.0 0.5 1.0
ax.set_xticks([250, 500, 750, 1000, 1250])
ax.set_xticklabels(["250", "500", "750", "1000", "1250"])

# ---------- EJE Y ----------
# Marcas en pasos de 0.05 desde 0.2 hasta 0.65
y_ticks = [0.35, 0.45, 0.55, 0.65]
ax.set_yticks(y_ticks)

# Rango del eje y desde 0.2 hasta 0.65
ax.set_ylim(0.35, 0.65)
# Esto imprime las etiquetas en forma 10⁻⁷, 10⁻⁶, etc.
ax.set_yticklabels([f"{ytick:.2f}" for ytick in y_ticks])
    
ax.plot(
    df_resultados["Numero"],
    df_resultados["V_set_derivada_V"],
    "o",
    color="blue",
    label="Max. current derivative det.",
)

ax.plot(
    df_resultados["Numero"],
    df_resultados["V_set_elbow_V"],
    "^",
    color="red",
    label="Max. separation from a straight line",
)

ax.plot(
#     df_resultados_sim["Numero"] + 854,
#     df_resultados_sim["V_creacion_1"],
#     "D",
#     color="green",
#     label="First filament creation voltage",
# )

# ax.plot(
#     df_resultados_sim["Numero"] + 854,
#     df_resultados_sim["V_creacion_2"],
#     "s",
#     label="Second filament creation voltage",
# )

# ax.plot(
#     df_resultados_sim["Numero"] + 854,
#     df_resultados_sim["V_creacion_3"],
#     "P",
#     label="All filament creation voltage",
)

ax.set_xlabel(r"Curve number", fontsize=40)
ax.set_ylabel(r"Set voltage (\si{\volt})", fontsize=40)
ax.legend(
    labelspacing=0.3,
    handletextpad=0.2,
    handlelength=1.0,
    borderaxespad=0.2,
    loc="best",
)
# ax.set_title("Voltaje de Set experimental vs Simulación")
fig.savefig(
    str(Path.cwd() / "Datos_Experimentales" / "V_Set" ) + "/V_set_experimental.pdf",
    dpi=300,
    bbox_inches="tight",
)  # type: ignore
plt.close(fig)